# 02 — Model Training

**Zar3y — Crop Disease Detection**

This notebook covers:
1. Loading preprocessed data with augmentation pipeline
2. Building MobileNetV3-Small with custom classifier head
3. Phase 1: Freeze backbone, train for 10 epochs
4. Phase 2: Unfreeze last 30 layers, fine-tune for 10 epochs (LR/10)
5. Early stopping on val_loss (patience=5)
6. Saving best checkpoint to `models/best_model.keras`

> **Run this notebook on Google Colab (free GPU tier)**

In [ ]:
import os
import json
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# -----------------------------
# Config
# -----------------------------
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

BASE_PATH = r"D:\Zar3y-Explainable-Crop-Disease-Detection-for-Smallholder-Farmers"

TRAIN_DIR = os.path.join(BASE_PATH, "data", "split_data", "train")
VAL_DIR   = os.path.join(BASE_PATH, "data", "split_data", "val")

MODEL_PATH   = os.path.join(BASE_PATH, "models", "best_model.keras")
HISTORY_PATH = os.path.join(BASE_PATH, "outputs", "training_history.json")

os.makedirs(os.path.join(BASE_PATH, "models"),  exist_ok=True)
os.makedirs(os.path.join(BASE_PATH, "outputs"), exist_ok=True)

print("Config loaded")

✅ Config loaded


In [4]:
# -----------------------------
# Load datasets
# -----------------------------
train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

val_ds = image_dataset_from_directory(
    VAL_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

print("Classes:", train_ds.class_names)
NUM_CLASSES = len(train_ds.class_names)
print(f"Number of classes: {NUM_CLASSES}")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)

Found 7886 files belonging to 10 classes.
Found 1692 files belonging to 10 classes.
Classes: ['Corn_(maize)___Common_rust_', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___healthy']
Number of classes: 10


In [5]:
# -----------------------------
# Data augmentation
# -----------------------------
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
    layers.RandomZoom(0.2),
])

print("✅ Augmentation pipeline ready")

✅ Augmentation pipeline ready


In [6]:
# -----------------------------
# Build model
# -----------------------------
def build_model(num_classes):
    inputs = tf.keras.Input(shape=(224, 224, 3))

    x = data_augmentation(inputs)
    x = preprocess_input(x)

    base_model = MobileNetV3Small(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = False

    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    return model, base_model


model, base_model = build_model(NUM_CLASSES)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         5,770 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 944,890 (3.60 MB)

 Trainable params: 5,770 (22.54 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [7]:
# -----------------------------
# Class weights
# -----------------------------
def compute_class_weights(train_ds):
    labels = []
    for _, y in train_ds:
        labels.extend(y.numpy().tolist())

    labels = np.array(labels)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(labels),
        y=labels
    )
    return dict(enumerate(weights))


print("Computing class weights...")
class_weights = compute_class_weights(train_ds)
print("Class Weights:", class_weights)

Computing class weights...
Class Weights: {0: np.float64(0.9455635491606714), 1: np.float64(1.1314203730272596), 2: np.float64(0.762669245647969), 3: np.float64(1.1265714285714286), 4: np.float64(1.1265714285714286), 5: np.float64(7.439622641509434), 6: np.float64(1.1265714285714286), 7: np.float64(0.5902694610778443), 8: np.float64(1.184084084084084), 9: np.float64(0.7085354896675652)}


In [8]:
# -----------------------------
# Callbacks
# -----------------------------
def get_callbacks():
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),
        ModelCheckpoint(
            MODEL_PATH,
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=2
        )
    ]

print("✅ Callbacks ready")

✅ Callbacks ready


In [9]:
# -----------------------------
# Phase 1 — Freeze backbone, train head
# -----------------------------
def train_phase1(model, train_ds, val_ds, epochs, class_weights):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=get_callbacks(),
        class_weight=class_weights
    )
    return history


print("🚀 Phase 1 Training...")
history1 = train_phase1(model, train_ds, val_ds, epochs=10, class_weights=class_weights)

🚀 Phase 1 Training...
Epoch 1/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step - accuracy: 0.4691 - loss: 1.6190
Epoch 1: val_loss improved from None to 0.47895, saving model to D:\Zar3y-Explainable-Crop-Disease-Detection-for-Smallholder-Farmers\models\best_model.keras
247/247 ━━━━━━━━━━━━━━━━━━━━ 77s 264ms/step - accuracy: 0.6498 - loss: 1.0882 - val_accuracy: 0.8712 - val_loss: 0.4789 - learning_rate: 0.0010
Epoch 2/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step - accuracy: 0.8446 - loss: 0.4930
Epoch 2: val_loss improved from 0.47895 to 0.34739, saving model to D:\Zar3y-Explainable-Crop-Disease-Detection-for-Smallholder-Farmers\models\best_model.keras
247/247 ━━━━━━━━━━━━━━━━━━━━ 75s 294ms/step - accuracy: 0.8592 - loss: 0.4537 - val_accuracy: 0.9025 - val_loss: 0.3474 - learning_rate: 0.0010
Epoch 3/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step - accuracy: 0.8841 - loss: 0.3576
Epoch 3: val_loss improved from 0.34739 to 0.26112, saving model to D:\Zar3y-Explainable-Crop-Disease-Detecti

In [11]:
# -----------------------------
# Phase 2 — Unfreeze last 30 layers, fine-tune
# -----------------------------
def train_phase2(model, base_model, train_ds, val_ds, epochs, class_weights):
    base_model.trainable = True

    for layer in base_model.layers[:-30]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=get_callbacks(),
        class_weight=class_weights
    )
    return history


print("🚀 Phase 2 Fine-tuning...")
history2 = train_phase2(model, base_model, train_ds, val_ds, epochs=10, class_weights=class_weights)

🚀 Phase 2 Fine-tuning...
Epoch 1/10


247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step - accuracy: 0.8485 - loss: 0.5319
Epoch 1: val_loss improved from None to 0.46138, saving model to D:\Zar3y-Explainable-Crop-Disease-Detection-for-Smallholder-Farmers\models\best_model.keras
247/247 ━━━━━━━━━━━━━━━━━━━━ 93s 317ms/step - accuracy: 0.8998 - loss: 0.3163 - val_accuracy: 0.8481 - val_loss: 0.4614 - learning_rate: 1.0000e-04
Epoch 2/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 412ms/step - accuracy: 0.9434 - loss: 0.1623
Epoch 2: val_loss improved from 0.46138 to 0.31433, saving model to D:\Zar3y-Explainable-Crop-Disease-Detection-for-Smallholder-Farmers\models\best_model.keras
247/247 ━━━━━━━━━━━━━━━━━━━━ 117s 459ms/step - accuracy: 0.9498 - loss: 0.1425 - val_accuracy: 0.8995 - val_loss: 0.3143 - learning_rate: 1.0000e-04
Epoch 3/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 358ms/step - accuracy: 0.9561 - loss: 0.1213
Epoch 3: val_loss improved from 0.31433 to 0.23232, saving model to D:\Zar3y-Explainable-Crop-Disease-Detection-for-Smallholder-Farme

In [12]:
# -----------------------------
# Combine & save training history
# -----------------------------
combined = {}
for k in history1.history:
    combined[k] = history1.history[k] + history2.history[k]

# Convert numpy types → python native types
for k in combined:
    combined[k] = [float(x) for x in combined[k]]

with open(HISTORY_PATH, "w") as f:
    json.dump(combined, f, indent=4)

print("\n✅ Training completed!")
print(f"Model saved at:   {MODEL_PATH}")
print(f"History saved at: {HISTORY_PATH}")


✅ Training completed!
Model saved at:   D:\Zar3y-Explainable-Crop-Disease-Detection-for-Smallholder-Farmers\models\best_model.keras
History saved at: D:\Zar3y-Explainable-Crop-Disease-Detection-for-Smallholder-Farmers\outputs\training_history.json
